In [29]:
!pip install langchain-core langchain-openai langchain

In [30]:
import os
from langchain_openai import ChatOpenAI
try:
  from google.colab import userdata
  OPENROUTER_API_KEY =   userdata.get("OPENROUTER_API_KEY")
except Exception:
  OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")


if not OPENROUTER_API_KEY:
  raise ValueError("OPENROUTER_API_KEY not found in environment.")


In [31]:
llm = ChatOpenAI(
model = "nvidia/nemotron-3-ultra-550b-a55b:free",
api_key = OPENROUTER_API_KEY,
base_url="https://openrouter.ai/api/v1",
temperature= 0.7,
)
print("LLM object created:",llm.model_name)

LLM object created: nvidia/nemotron-3-ultra-550b-a55b:free


In [32]:
response= llm.invoke("Say hello exactly in five words")

In [33]:
from langchain_core.prompts import PromptTemplate

In [34]:
template = PromptTemplate.from_template("write a one sentence summary of topic for a beginner:{topic} ")
filled = template.format(topic="maths")
print(filled)

write a one sentence summary of topic for a beginner:maths 


In [35]:
#langchain | pipe operator
from langchain_core.output_parsers import StrOutputParser

In [36]:
parser = StrOutputParser()
chain = template | llm | parser
result = chain.invoke({"topic":"maths"})
print(result)

Mathematics is the universal language of patterns, logic, and structures that uses numbers and symbols to measure, model, and make sense of the world around us.


In [37]:
from langchain_core.prompts import ChatPromptTemplate

In [38]:
chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a concise assistant. Always answer in one short sentence."),
    ("user", "{question}"),
])

In [39]:
filled_message = chat_template.format_messages(question="What causes seasons on Earth?")
for m in filled_message:
  print(m.type,"-",m.content)

system - You are a concise assistant. Always answer in one short sentence.
human - What causes seasons on Earth?


In [40]:
print(filled_message)

[SystemMessage(content='You are a concise assistant. Always answer in one short sentence.', additional_kwargs={}, response_metadata={}), HumanMessage(content='What causes seasons on Earth?', additional_kwargs={}, response_metadata={})]


In [41]:
chat_chain = chat_template | llm | parser
answer = chat_chain.invoke({"question":"what causes seasons on Earth"})
print(answer)

Seasons result from Earth's ~23.5° axial tilt causing different hemispheres to receive varying sunlight angles and day lengths as the planet orbits the Sun.


In [42]:
multi_template = ChatPromptTemplate.from_messages([
    ("system", "You are a {persona}."),
    ("user", "Explain {topic} in one sentence."),
])

multi_chain = multi_template | llm | parser
result = multi_chain.invoke({"persona": "patient LLM Trainer teacher", "topic": "api calling"})
print(result)


API calling is the process where a client application sends a structured request to a server's specific endpoint to trigger an action or retrieve data, then receives a structured response.


In [ ]:
#chaining the output of llm to another llm call (sequential chain)

In [49]:
premise_template=ChatPromptTemplate.from_messages([
    ("system", "You are a creative writing assistant."),
    ("user", "In one sentence, invent a story premise in the {genre} genre."),

])

In [52]:
premise_chain = premise_template | llm | parser

In [45]:
opening_line_template = ChatPromptTemplate.from_messages([
    ("system", "You are a creative writing assistant."),
    ("user", "Given this story premise: \"{premise}\", write one compelling opening line for the story."),
])

In [46]:
opening_line_chain = opening_line_template | llm | parser

In [47]:
premise = premise_chain.invoke({"genre": "horror"})
print(premise)

A lullaby monitor picks up a second, deeper voice harmonizing with the mother’s song from inside the nursery—though the baby hasn't made a sound in weeks.


In [56]:
opening  = opening_line_chain.invoke({"premise": premise})
print(opening)

The monitor didn't catch the baby’s breath—it caught the baritone harmonizing *under* hers, vibrating from a crib that had held only silence for forty-two days.


In [57]:
def story_pipeline(genre):
  premise = premise_chain.invoke({"genre":genre})
  opening_line = opening_line_chain.invoke({"premise":premise})
  return {"genre":genre,"premise":premise,"opening_line":opening_line}

result = story_pipeline("Sci-fi")
print(result)

{'genre': 'Sci-fi', 'premise': "When a deep-space salvage crew awakens an ancient alien AI that rewrites their ship's physics to simulate a perfect afterlife for their dead loved ones, they must choose between escaping the simulation to warn humanity of an impending invasion or staying forever in a paradise built from their own memories.", 'opening_line': 'The salvage lights of the *USC Meridian* didn’t cut through the dark—they *rewrote* it, swapping the vacuum of Sector 9 for the smell of rain on hot asphalt and the sound of my daughter’s laugh, recorded in a memory I’d sold to the company store twenty years ago.'}


In [61]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
except Exception:
    OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")

llm = ChatOpenAI(
model = "nvidia/nemotron-3-ultra-550b-a55b:free",
api_key = OPENROUTER_API_KEY,
base_url="https://openrouter.ai/api/v1",
temperature= 0.7,
)
parser = StrOutputParser()

ingredient_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful cooking assistant."),
    ("user", "Name exactly one key ingredient needed to cook {dish}. Reply with only the ingredient name."),
])
ingredient_chain = ingredient_template | llm | parser

fact_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful cooking assistant."),
    ("user", "Give one short, fun fact about {ingredient}."),
])
fact_chain = fact_template | llm | parser

def dish_fact_pipeline(dish):
    ingredient = ingredient_chain.invoke({"dish": dish })
    fact = fact_chain.invoke({"ingredient": ingredient })
    return {"dish": dish, "ingredient": ingredient, "fact": fact}

print(dish_fact_pipeline("Tomato curry"))

{'dish': 'Tomato curry', 'ingredient': 'Tomatoes', 'fact': 'Tomatoes were once put on trial in the US—literally. In 1820, Colonel Robert Gibbon Johnson allegedly ate a basket of them on the Salem, New Jersey courthouse steps to prove they weren\'t poisonous (a common belief at the time), while a doctor waited to treat his "certain death." He survived, and the tomato\'s reputation was saved.'}
